# Embed ASR/caption trước bằng Qwen vLLM

Notebook này dùng cùng quy ước range với `extract_embedding.ipynb`: chọn `DATASET`, `START_VIDEO_ID`, `END_VIDEO_ID` (bao gồm hai đầu), lọc đúng video từ Drive, rồi dùng Qwen3-VL-Embedding-2B qua vLLM. Ví dụ V3C `00011` đến `00020`; LHE `LHE11` đến `LHE20`. Mỗi segment lưu thành vector riêng theo ordinal JSON: `asr/V3C/00001/000000.npy` hoặc `caption/V3C/00001/000000.npy`. Caption không có keyframe map sẽ bị loại trước khi embed.

Cần bật Internet + GPU Kaggle, điền folder ID cho dataset và folder output. ASR JSON, caption JSON và map keyframe đều cần để tạo mapping theo shot. Notebook chỉ embed dữ liệu JSON đã có; không chạy Whisper hay model tạo caption.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, time, urllib.request, urllib.error, shutil

REPO_URL = 'https://github.com/iamtro-2006/event-retrieval-system.git'
REPO_REF = 'vbs-dev'  # Nhánh chứa scripts/embed_segment_data.py tương thích với notebook này
REPO = Path('/kaggle/working/event-retrieval-system')
BACKEND = REPO/'backend'
WORK = Path('/kaggle/working/ers-segment-embedding')
WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK/'data'
OUTPUT_ROOT = WORK/'embeddings_qwen3_vl_embedding_2b_vllm'

# Cùng quy ước với extract_embedding.ipynb; END_VIDEO_ID được tính cả.
DATASET = 'V3C'  # 'V3C' hoặc 'LHE'
START_VIDEO_ID = 1
END_VIDEO_ID = 10
DATASET_FOLDERS = {
    'V3C': {
        'asr_folder_id': '',
        'caption_folder_id': '',
        'map_folder_id': '',
    },
    # 'LHE': {'asr_folder_id':'', 'caption_folder_id':'', 'map_folder_id':''},
}
SOURCES_TO_EMBED = ['asr', 'caption']  # Chọn một source hoặc cả hai.
OUTPUT_FOLDER_ID = ''  # Drive folder gốc; bên trong lưu <source>/<DATASET>/<video_id>/<segment_number>.npy.
QWEN_MODEL = 'Qwen/Qwen3-VL-Embedding-2B'
SEGMENT_BATCH_SIZE = 8
VLLM_PORT = 8002
RCLONE_CONFIG_SOURCE = Path('/kaggle/input/datasets/nguyennhattan121106/rclone-pro-pro/rclone.conf')

if DATASET not in {'V3C', 'LHE'}:
    raise ValueError("DATASET phải là 'V3C' hoặc 'LHE'")
max_id = 99 if DATASET == 'LHE' else 99999
if (type(START_VIDEO_ID) is not int or type(END_VIDEO_ID) is not int
        or not 1 <= START_VIDEO_ID <= END_VIDEO_ID <= max_id):
    raise ValueError(f'Cần 1 <= START_VIDEO_ID <= END_VIDEO_ID <= {max_id} cho {DATASET}')
def video_name(number):
    return f'LHE{number:02d}' if DATASET == 'LHE' else f'{number:05d}'
SELECTED_VIDEO_IDS = [video_name(number) for number in range(START_VIDEO_ID, END_VIDEO_ID + 1)]
DATASETS = {DATASET.lower(): DATASET_FOLDERS[DATASET]}
if not OUTPUT_FOLDER_ID:
    raise ValueError('Điền OUTPUT_FOLDER_ID trước khi chạy')
if not SOURCES_TO_EMBED or set(SOURCES_TO_EMBED)-{'asr','caption'}:
    raise ValueError('SOURCES_TO_EMBED chỉ nhận asr/caption')
folders = DATASETS[DATASET.lower()]
if not folders.get('caption_folder_id') or not folders.get('map_folder_id'):
    raise ValueError(f'Thiếu Drive folder ID caption/map_keyframes cho {DATASET}')
if 'asr' in SOURCES_TO_EMBED and not folders.get('asr_folder_id'):
    raise ValueError(f'Thiếu Drive folder ID ASR cho {DATASET}')
print(f'Processing {DATASET} video IDs {START_VIDEO_ID}–{END_VIDEO_ID}: {SELECTED_VIDEO_IDS[0]}…{SELECTED_VIDEO_IDS[-1]}', flush=True)

if not REPO.exists():
    subprocess.run(['git','clone',REPO_URL,str(REPO)],check=True)
if REPO_REF:
    subprocess.run(['git','-C',str(REPO),'checkout',REPO_REF],check=True)
assert (BACKEND/'scripts/embed_segment_data.py').is_file(), 'Repo chưa có scripts/embed_segment_data.py'
os.chdir(BACKEND)
sys.path.insert(0,str(BACKEND))
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','httpx','PyYAML','python-dotenv','uv'],check=True)

rclone = shutil.which('rclone')
if not rclone:
    import zipfile, urllib.request as urlrequest
    release='v1.75.2'; archive=WORK/f'rclone-{release}-linux-amd64.zip'
    urlrequest.urlretrieve(f'https://github.com/rclone/rclone/releases/download/{release}/{archive.name}',archive)
    with zipfile.ZipFile(archive) as z:
        member=next(x for x in z.namelist() if x.endswith('/rclone'))
        binary=WORK/'rclone'; binary.write_bytes(z.read(member)); binary.chmod(0o755)
    rclone=str(binary)

rclone_config=WORK/'rclone.conf'
config_text=None
try:
    from kaggle_secrets import UserSecretsClient
    config_text=UserSecretsClient().get_secret('RCLONE_CONFIG')
except Exception:
    # Secret is optional when rclone.conf is attached as Kaggle Input.
    pass
if config_text:
    rclone_config.write_text(config_text,encoding='utf-8')
elif RCLONE_CONFIG_SOURCE.is_file():
    shutil.copy2(RCLONE_CONFIG_SOURCE,rclone_config)
else:
    raise FileNotFoundError(f'Không thấy rclone config file: {RCLONE_CONFIG_SOURCE}; Kaggle Secret RCLONE_CONFIG cũng có thể dùng thay thế')
rclone_config.chmod(0o600)
subprocess.run([rclone,'--config',str(rclone_config),'listremotes'],check=True)

VLLM_VENV=WORK/'vllm-venv'
if not (VLLM_VENV/'bin/vllm').exists():
    subprocess.run([sys.executable,'-m','uv','venv','--python','3.12',str(VLLM_VENV)],check=True)
    subprocess.run([sys.executable,'-m','uv','pip','install','--python',str(VLLM_VENV/'bin/python'),
                    '--torch-backend=auto','-r',str(BACKEND/'requirements-vllm.txt')],check=True)

In [ ]:
# Tải đúng JSON/CSV trong range đã chọn, giữ nguyên tên video.
def list_drive_files(folder_id):
    result=subprocess.run([rclone,'--config',str(rclone_config),'--drive-root-folder-id',folder_id,
                          'lsf','iamtro:','--files-only','--recursive','--format','p'],
                         check=True,capture_output=True,text=True)
    return [line.strip() for line in result.stdout.splitlines() if line.strip()]

def download_folder(folder_id, target, suffix, video_ids):
    target.mkdir(parents=True,exist_ok=True)
    expected={f'{video_id}{suffix}' for video_id in video_ids}
    matches=[name for name in list_drive_files(folder_id)
             if Path(name).name in expected and Path(name).suffix.lower()==suffix]
    by_name={}
    for relative in matches:
        rel=Path(relative)
        if rel.is_absolute() or '..' in rel.parts:
            raise ValueError(f'Invalid Drive path: {relative}')
        by_name.setdefault(rel.name,[]).append(relative)
    missing=sorted(expected-set(by_name))
    duplicates={name:paths for name,paths in by_name.items() if len(paths)!=1}
    if missing or duplicates:
        raise RuntimeError(f'Range {START_VIDEO_ID}-{END_VIDEO_ID} thiếu/trùng {suffix}: missing={missing[:10]}, duplicates={duplicates}')
    for name in sorted(expected):
        relative=by_name[name][0]
        destination=target/name
        subprocess.run([rclone,'--config',str(rclone_config),'--drive-root-folder-id',folder_id,
                        'copyto',f'iamtro:{relative}',str(destination)],check=True)
    print(f'{target}: downloaded {len(expected)} {suffix} files for selected range',flush=True)

for dataset, folders in DATASETS.items():
    upper=dataset.upper()
    for input_kind in ['asr','captions','map_keyframes']:
        stale=DATA/input_kind/upper
        if stale.exists():
            shutil.rmtree(stale)
    for source in SOURCES_TO_EMBED:
        stale=OUTPUT_ROOT/source/upper
        if stale.exists():
            shutil.rmtree(stale)
    if 'asr' in SOURCES_TO_EMBED:
        download_folder(folders['asr_folder_id'],DATA/'asr'/upper,'.json',SELECTED_VIDEO_IDS)
    download_folder(folders['caption_folder_id'],DATA/'captions'/upper,'.json',SELECTED_VIDEO_IDS)
    download_folder(folders['map_folder_id'],DATA/'map_keyframes'/upper,'.csv',SELECTED_VIDEO_IDS)

In [ ]:
# Chạy Qwen vLLM; lưu một vector riêng mỗi segment theo ordinal JSON, ví dụ asr/V3C/00001/000000.npy.
QWEN_VLLM_URL=f'http://127.0.0.1:{VLLM_PORT}/v1'
os.environ['QWEN_VLLM_URL']=QWEN_VLLM_URL
os.environ['QWEN_EMBEDDING_ENGINE']='vllm'
VLLM_LOG=WORK/'vllm.log'
log_handle=VLLM_LOG.open('a',encoding='utf-8')
process=subprocess.Popen([str(VLLM_VENV/'bin/vllm'),'serve',QWEN_MODEL,'--runner','pooling',
                          '--max-model-len','8192','--host','127.0.0.1','--port',str(VLLM_PORT)],
                         stdout=log_handle,stderr=subprocess.STDOUT)
try:
    deadline=time.monotonic()+600
    while time.monotonic()<deadline:
        if process.poll() is not None:
            raise RuntimeError(f'vLLM exited; inspect {VLLM_LOG}')
        try:
            with urllib.request.urlopen(QWEN_VLLM_URL+'/models',timeout=3) as response:
                models=json.load(response).get('data',[])
            if any(item.get('id')==QWEN_MODEL for item in models):
                break
        except (OSError,ValueError,urllib.error.URLError):
            pass
        time.sleep(3)
    else:
        raise TimeoutError(f'Qwen vLLM chưa sẵn sàng: {QWEN_VLLM_URL}')

    for dataset in DATASETS:
        for source in SOURCES_TO_EMBED:
            subprocess.run([sys.executable,'-u','-m','scripts.embed_segment_data',
                            '--dataset',dataset,'--source',source,'--data-root',str(DATA),
                            '--output-root',str(OUTPUT_ROOT),'--batch-size',str(SEGMENT_BATCH_SIZE),
                            '--video-ids',*SELECTED_VIDEO_IDS],
                           check=True,env=os.environ.copy())
finally:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill(); process.wait()
    log_handle.close()

In [ ]:
import numpy as np
from src.retrieval.segment_data import load_segment_files, SegmentMaps
# Kiểm tra từng segment đã được embed riêng và caption đã lọc theo map.
for dataset in DATASETS:
    for source in SOURCES_TO_EMBED:
        source_files=dict(load_segment_files(DATA,dataset,source))
        if source=='caption':
            all_records=[record for records in source_files.values() for record in records]
            maps=SegmentMaps(DATA,dataset,'caption',all_records)
            source_files={video_id:[record for record in records if maps.forward.get(record['segment_id'])]
                          for video_id,records in source_files.items()}
        for video_id in SELECTED_VIDEO_IDS:
            records=source_files.get(video_id,[])
            video_dir=OUTPUT_ROOT/source/dataset.upper()/video_id
            expected={f"{int(record['segment_id'].rsplit(':',1)[-1]):06d}.npy" for record in records}
            actual={f.name for f in video_dir.glob('*.npy')} if video_dir.is_dir() else set()
            assert actual==expected, (source,video_id,sorted(expected-actual)[:10],sorted(actual-expected)[:10])
            for name in expected:
                vector=np.load(video_dir/name,mmap_mode='r',allow_pickle=False)
                assert vector.shape==(2048,), (video_dir/name,vector.shape)
        print(f"{dataset}/{source}: verified individual segment vectors",flush=True)

In [ ]:
# Upload từng vector segment theo range.
assert OUTPUT_ROOT.is_dir(), f'Embedding output missing: {OUTPUT_ROOT}'
for source in SOURCES_TO_EMBED:
    for video_id in SELECTED_VIDEO_IDS:
        video_dir=OUTPUT_ROOT/source/DATASET/video_id
        for artifact in sorted(video_dir.glob('*.npy')):
            subprocess.run([rclone,'--config',str(rclone_config),'--drive-root-folder-id',OUTPUT_FOLDER_ID,
                            'copyto',str(artifact),
                            f'iamtro:{source}/{DATASET}/{video_id}/{artifact.name}'],check=True)
print(f'Uploaded {DATASET} {START_VIDEO_ID}–{END_VIDEO_ID} segment vector files to Drive folder:',OUTPUT_FOLDER_ID)